In [60]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# Imports and Configuratiion

In [61]:
# Import Libraries

import numpy as np
import pandas as pd
import random
import os


import matplotlib.pyplot as plt
import seaborn as sns

import re
import string


from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer

from scipy.sparse import hstack, csr_matrix


import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import accuracy_score
from tqdm import tqdm

import warnings
warnings.filterwarnings("ignore")

In [62]:
# Reproducibility

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

In [63]:
# Device

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

cuda


# Load Dataset

In [64]:
# Load Dataset

train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
sample = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")

# Exploratory Data Analysis

In [65]:
print(train.shape)
print(test.shape)

train.head()

(2000, 8)
(500, 7)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [66]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      2000 non-null   int64 
 1   prompt  2000 non-null   object
 2   A       2000 non-null   object
 3   B       2000 non-null   object
 4   C       2000 non-null   object
 5   D       2000 non-null   object
 6   E       2000 non-null   object
 7   answer  2000 non-null   object
dtypes: int64(1), object(7)
memory usage: 125.1+ KB


## Checking missing values

In [67]:
print(train.isnull().sum())

print(test.isnull().sum())

id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0
dtype: int64
id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
dtype: int64


## Fill missing values, encode labels, and clean the text columns

In [68]:
columns = ["prompt","A","B","C","D","E"]

for col in columns:
    train[col] = train[col].fillna("")
    test[col] = test[col].fillna("")

In [69]:
# Encoding the label column 
label_encoder = LabelEncoder()

train["label"] = label_encoder.fit_transform(train["answer"])

In [70]:
print(label_encoder.classes_)

['A' 'B' 'C' 'D' 'E']


In [71]:
# Text Cleaning

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"\s+", " ", text)
    text = text.strip()
    return text

In [72]:
for col in columns:
    train[col] = train[col].apply(clean_text)
    test[col] = test[col].apply(clean_text)

In [73]:
train.head()

,id,prompt,A,B,C,D,E,answer,label
0,1,pick the best possible answer: what is martin ...,martin heidegger believes that humans exist wi...,martin heidegger believes that humans do not e...,martin heidegger does not believe in the exist...,martin heidegger believes that the relationshi...,martin heidegger believes that time is an illu...,B,1
1,2,what is accelerator-based light-ion fusion?,accelerator-based light-ion fusion is a techni...,accelerator-based light-ion fusion is a techni...,accelerator-based light-ion fusion is a techni...,accelerator-based light-ion fusion is a techni...,accelerator-based light-ion fusion is a techni...,A,0
2,3,determine the correct option: what is the term...,blueshifting,redshifting,reddening,whitening,yellowing,C,2
3,4,select the most accurate option: what is marti...,martin heidegger believes that humans exist wi...,martin heidegger believes that humans do not e...,martin heidegger does not believe in the exist...,martin heidegger believes that the relationshi...,martin heidegger believes that time is an illu...,B,1
4,5,identify the correct statement: what is the co...,"simultaneity is relative, meaning that two eve...","simultaneity is relative, meaning that two eve...","simultaneity is absolute, meaning that two eve...",simultaneity is a concept that applies only to...,simultaneity is a concept that applies only to...,A,0


# Split training data into train and validation sets

In [74]:
train_df, valid_df = train_test_split(
    train,
    test_size=0.2,
    random_state=SEED,
    stratify=train["label"]
)

train_df = train_df.reset_index(drop=True)
valid_df = valid_df.reset_index(drop=True)

print("Training Questions :", len(train_df))
print("Validation Questions :", len(valid_df))

Training Questions : 1600
Validation Questions : 400


**Expand each question into one row per option, for binary classification.**

In [75]:
# Create Prompt-Option Pairs

OPTION_COLUMNS = ["A", "B", "C", "D", "E"]

def create_pairs(df, training=True):

    texts = []
    labels = []
    question_ids = []
    option_names = []
    for _, row in df.iterrows():
        prompt = row["prompt"]
        for option in OPTION_COLUMNS:
            
            # Prompt + option text
            pair = prompt + " [SEP] " + row[option]
            texts.append(pair)
            option_names.append(option)
            question_ids.append(row["id"])

            if training:
                labels.append(1 if row["answer"] == option else 0)

    if training:

        return pd.DataFrame({
            "id": question_ids,
            "text": texts,
            "option": option_names,
            "label": labels
        })

    return pd.DataFrame({
        "id": question_ids,
        "text": texts,
        "option": option_names
    })

In [76]:
train_pairs = create_pairs(train_df)

valid_pairs = create_pairs(valid_df)

test_pairs = create_pairs(test, training=False)

In [77]:
train_pairs.head(10)

,id,text,option,label
0,387,select the most accurate option: what is modif...,A,0
1,387,select the most accurate option: what is modif...,B,0
2,387,select the most accurate option: what is modif...,C,1
3,387,select the most accurate option: what is modif...,D,0
4,387,select the most accurate option: what is modif...,E,0
5,1901,determine the correct option: what is lorentz ...,A,0
6,1901,determine the correct option: what is lorentz ...,B,0
7,1901,determine the correct option: what is lorentz ...,C,0
8,1901,determine the correct option: what is lorentz ...,D,0
9,1901,determine the correct option: what is lorentz ...,E,1


In [78]:
print(train_pairs.shape)
print(valid_pairs.shape)
print(test_pairs.shape)

(8000, 4)
(2000, 4)
(2500, 3)


# Build word-level and character-level TF-IDF features and combine them

In [79]:
# Word TF-IDF

word_vectorizer = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1,2),
    min_df=2,
    sublinear_tf=True,
    stop_words="english"
)

In [80]:
word_vectorizer.fit(train_pairs["text"])

TfidfVectorizer(max_features=50000, min_df=2, ngram_range=(1, 2),
                stop_words='english', sublinear_tf=True)

In [81]:
X_train_word = word_vectorizer.transform(train_pairs["text"])

X_valid_word = word_vectorizer.transform(valid_pairs["text"])

X_test_word = word_vectorizer.transform(test_pairs["text"])

In [82]:
print(X_train_word.shape)

(8000, 11105)


In [83]:
# Character TF-IDF

char_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3,5),
    max_features=30000,
    sublinear_tf=True
)

In [84]:
char_vectorizer.fit(train_pairs["text"])

TfidfVectorizer(analyzer='char_wb', max_features=30000, ngram_range=(3, 5),
                sublinear_tf=True)

In [85]:
X_train_char = char_vectorizer.transform(train_pairs["text"])

X_valid_char = char_vectorizer.transform(valid_pairs["text"])

X_test_char = char_vectorizer.transform(test_pairs["text"])

In [86]:
# Combine Features

X_train = hstack([X_train_word, X_train_char])

X_valid = hstack([X_valid_word, X_valid_char])

X_test = hstack([X_test_word, X_test_char])

In [87]:
print(X_train.shape)

(8000, 35021)


In [88]:
y_train = train_pairs["label"].values

y_valid = valid_pairs["label"].values

In [89]:
print(y_train[:20])

[0 0 1 0 0 0 0 0 0 1 0 0 0 0 1 0 0 0 1 0]


In [90]:
print(train_pairs["label"].value_counts())

label
0    6400
1    1600
Name: count, dtype: int64


# Wrap features into a PyTorch Dataset and create DataLoaders

In [91]:
# Custom Dataset

class MCQDataset(Dataset):

    def __init__(self, X, y=None):

        self.X = X
        self.y = y

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):

        # Convert sparse row to dense
        features = torch.FloatTensor(
            self.X[idx].toarray().squeeze()
        )
        if self.y is not None:
            label = torch.FloatTensor([self.y[idx]])
            return features, label

        return features

In [92]:
train_dataset = MCQDataset(X_train, y_train)

valid_dataset = MCQDataset(X_valid, y_valid)

test_dataset = MCQDataset(X_test)

In [93]:
BATCH_SIZE = 128

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2
)

**Compute a positive class weight to handle class imbalance.**

In [94]:
positive = np.sum(y_train == 1)

negative = np.sum(y_train == 0)

print(positive, negative)

1600 6400


In [95]:
pos_weight = torch.tensor(
    [negative / positive],
    dtype=torch.float32
).to(device)

print(pos_weight)

tensor([4.], device='cuda:0')


# Define the neural network, loss, optimizer, and scheduler

In [96]:
# Deep Neural Network

class DeepMCQNet(nn.Module):

    def __init__(self, input_dim):

        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.40),
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.30),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.20),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        return self.network(x)

In [97]:
INPUT_DIM = X_train.shape[1]

model = DeepMCQNet(INPUT_DIM).to(device)

print(model)

DeepMCQNet(
  (network): Sequential(
    (0): Linear(in_features=35021, out_features=1024, bias=True)
    (1): BatchNorm1d(1024, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.4, inplace=False)
    (4): Linear(in_features=1024, out_features=512, bias=True)
    (5): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=512, out_features=128, bias=True)
    (9): ReLU()
    (10): Dropout(p=0.2, inplace=False)
    (11): Linear(in_features=128, out_features=1, bias=True)
  )
)


In [98]:
criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight
)

In [99]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

In [100]:
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2
)

In [101]:
print("Input Dimension :", INPUT_DIM)

print("Training Samples :", len(train_dataset))

print("Validation Samples :", len(valid_dataset))

print("Test Samples :", len(test_dataset))

Input Dimension : 35021
Training Samples : 8000
Validation Samples : 2000
Test Samples : 2500


# Training, validation, and the MAP@3 metric.

In [102]:
# Training Function

def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss = 0

    for features, labels in tqdm(loader):
        features = features.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(features)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    return running_loss / len(loader)

In [103]:
# Validation Function

def validate(model, loader, criterion):

    model.eval()
    running_loss = 0
    predictions = []
    labels_list = []

    with torch.no_grad():

        for features, labels in tqdm(loader):
            features = features.to(device)
            labels = labels.to(device)
            outputs = model(features)
            loss = criterion(outputs, labels)
            running_loss += loss.item()
            probs = torch.sigmoid(outputs)
            predictions.extend(probs.cpu().numpy())
            labels_list.extend(labels.cpu().numpy())

    return (
        running_loss / len(loader),
        np.array(predictions),
        np.array(labels_list)
    )

In [104]:
# MAP@3

def mapk(actual, predicted, k=3):

    score = 0.0
    for i, p in enumerate(predicted[:k]):

        if p == actual:
            score = 1 / (i + 1)
            break

    return score

In [105]:
def evaluate_map3(df_pairs, predictions):

    temp = df_pairs.copy()
    temp["score"] = predictions
    scores = []

    for question_id, group in temp.groupby("id"):
        group = group.sort_values(
            "score",
            ascending=False
        )
        predicted = group["option"].tolist()[:3]

        actual = group.loc[
            group["label"] == 1,
            "option"
        ].values[0]

        scores.append(
            mapk(actual, predicted)
        )

    return np.mean(scores)

**Train the model with early stopping and evaluate final MAP@3.**

In [106]:
EPOCHS = 20

best_loss = np.inf

patience = 3

counter = 0

In [107]:
# Training Loop

for epoch in range(EPOCHS):

    print(f"\nEpoch {epoch+1}/{EPOCHS}")

    train_loss = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer
    )

    valid_loss, predictions, labels = validate(
        model,
        valid_loader,
        criterion
    )

    scheduler.step(valid_loss)

    map3 = evaluate_map3(
        valid_pairs,
        predictions.flatten()
    )

    print(f"Train Loss : {train_loss:.4f}")

    print(f"Valid Loss : {valid_loss:.4f}")

    print(f"MAP@3 : {map3:.4f}")

    if valid_loss < best_loss:
        best_loss = valid_loss
        counter = 0
        torch.save(
            model.state_dict(),
            "best_model.pt"
        )

        print("Best model saved.")

    else:
        counter += 1
        print(f"EarlyStopping Counter : {counter}")
        if counter >= patience:
            print("Stopping Training")
            break


Epoch 1/20


100%|██████████| 16/16 [00:00<00:00, 34.22it/s]


Train Loss : 0.9247
Valid Loss : 0.5441
MAP@3 : 0.9800
Best model saved.

Epoch 2/20


100%|██████████| 16/16 [00:00<00:00, 33.24it/s]


Train Loss : 0.4163
Valid Loss : 0.2778
MAP@3 : 0.9975
Best model saved.

Epoch 3/20


...

Train Loss : 0.0979
Valid Loss : 0.1198
MAP@3 : 0.9975
EarlyStopping Counter : 1

Epoch 14/20


100%|██████████| 16/16 [00:00<00:00, 33.81it/s]


Train Loss : 0.0999
Valid Loss : 0.0545
MAP@3 : 0.9975
EarlyStopping Counter : 2

Epoch 15/20


100%|██████████| 16/16 [00:00<00:00, 35.70it/s]


Train Loss : 0.0717
Valid Loss : 0.0989
MAP@3 : 0.9975
EarlyStopping Counter : 3
Stopping Training


In [108]:
model.load_state_dict(
    torch.load("best_model.pt")
)

<All keys matched successfully>

In [109]:
valid_loss, predictions, labels = validate(
    model,
    valid_loader,
    criterion
)

map3 = evaluate_map3(
    valid_pairs,
    predictions.flatten()
)

print("Final MAP@3 :", map3)

100%|██████████| 16/16 [00:00<00:00, 35.16it/s]


Final MAP@3 : 0.9975


# Predict on test data and save the submission file

In [110]:
# Test Prediction

def predict(model, loader):

    model.eval()

    predictions = []

    with torch.no_grad():

        for features in tqdm(loader):

            features = features.to(device)
            outputs = model(features)
            probs = torch.sigmoid(outputs)
            predictions.extend(
                probs.cpu().numpy()
            )

    return np.array(predictions)

In [111]:
test_predictions = predict(
    model,
    test_loader
)

print(test_predictions.shape)

100%|██████████| 20/20 [00:00<00:00, 37.94it/s]

(2500, 1)


In [112]:
test_predictions = test_predictions.flatten()

In [113]:
test_pairs["score"] = test_predictions

In [114]:
test_pairs.head()

,id,text,option,score
0,1,pick the best possible answer: what is the rel...,A,0.963761
1,1,pick the best possible answer: what is the rel...,B,0.005274
2,1,pick the best possible answer: what is the rel...,C,0.000401
3,1,pick the best possible answer: what is the rel...,D,0.002631
4,1,pick the best possible answer: what is the rel...,E,0.003698


In [115]:
# Generate Submission

submission_predictions = []

for question_id, group in test_pairs.groupby("id"):

    group = group.sort_values(
        "score",
        ascending=False
    )

    top3 = group["option"].tolist()[:3]

    submission_predictions.append(
        " ".join(top3)
    )

In [116]:
submission = pd.DataFrame({

    "id": test["id"],

    "Prediction": submission_predictions

})

In [117]:
submission.to_csv(

    "submission.csv",

    index=False

)

print("Submission Saved Successfully!")

Submission Saved Successfully!


In [118]:
submission.head(10)

,id,Prediction
0,1,A B E
1,2,B E A
2,3,B A D
3,4,E C D
4,5,C A B
5,6,D C B
6,7,E A B
7,8,B E A
8,9,C D A
9,10,B E A
